# ============================================================
# Kaggle Notebook: DKT + Graph-DKT Retrain (GPU)
# ============================================================
# RUN IN ORDER. Each section is a separate cell.
# Accelerator recommendation: GPU T4 x2 or GPU T4 x1
# ============================================================


### CELL 0 — Install dependencies


In [ ]:
# Install torch-geometric so the Graph-DKT path is activated.
# The dense GCN uses pure PyTorch (einsum) under the hood, so heavy binary dependencies
# (like pyg_lib, torch_scatter, torch_sparse, torch_cluster) are NOT required.
# Kaggle already has a fully-functional PyTorch + CUDA pre-installed.
# Note: You MUST toggle 'Internet on' in the right panel under Settings to download packages!
try:
    import torch_geometric
    print("torch_geometric is already installed!")
except ImportError:
    print("Installing torch-geometric...")
    !pip install torch-geometric -q || echo "WARNING: Could not install torch-geometric. Check if Internet is toggled ON in the right panel under Settings. Falling back to pure PyTorch dense GCN mode."

!pip install scikit-learn pandas tqdm -q

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    try:
        print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    except AttributeError:
        props = torch.cuda.get_device_properties(0)
        mem = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
        print(f"VRAM: {mem / 1e9:.1f} GB")

### CELL 1 — Copy training.csv from Kaggle input


In [ ]:
# When you use Kaggle's "Add Data" → "Upload", the file
# lands in /kaggle/input/<dataset-name>/training.csv

import os, shutil

# Find training.csv in Kaggle input directory
input_dir = "/kaggle/input"
training_csv = None
for root, dirs, files in os.walk(input_dir):
    if "training.csv" in files:
        training_csv = os.path.join(root, "training.csv")
        break

if training_csv is None:
    print("ERROR: training.csv not found in Kaggle input!")
    print("Go to: Add data → Upload → select training.csv")
    print("Then re-run this cell.")
    import sys; sys.exit(1)

print(f"Found: {training_csv}")
print(f"Size: {os.path.getsize(training_csv) / 1e6:.1f} MB")

### CELL 2 — Clone repo and copy training data


In [ ]:
!git clone https://github.com/MannPatel1236/CP-Coach.git /kaggle/working/CP-Coach

# Copy training.csv into repo
data_dir = "/kaggle/working/CP-Coach/backend/data"
os.makedirs(data_dir, exist_ok=True)
shutil.copy(training_csv, os.path.join(data_dir, "training.csv"))

print(f"Copied training.csv to {data_dir}")
!wc -l /kaggle/working/CP-Coach/backend/data/training.csv

### CELL 3 — Verify data + quick stats


In [ ]:
%cd /kaggle/working/CP-Coach/backend

import sys
sys.path.insert(0, "/kaggle/working/CP-Coach/backend")

from data.topic_graph import CPTopicGraph
from training.train_dkt import load_csv_sequences

tg = CPTopicGraph()
# Let's load with max_seq_len=2000 matching our training configurations
seqs = load_csv_sequences("data/training.csv", tg, max_seq_len=2000)
print(f"Loaded {len(seqs)} user sequences (max_seq_len=2000)")

import numpy as np
lengths = np.array([len(s) for s in seqs])
print(f"Sequence length stats (after truncation):")
for p in [50, 75, 90, 95, 99, 100]:
    print(f"  p{p}: {np.percentile(lengths, p):.0f}")

### CELL 4 — DKT smoke test (3 epochs, 1 fold)


In [ ]:
%cd /kaggle/working/CP-Coach/backend
!python3 -m training.train_dkt \
  --data data/training.csv --model dkt \
  --epochs 3 --folds 1 --out weights/smoke.pt \
  --batch 16 --device cuda --max-seq-len 2000

### CELL 5 — DKT full retrain (5 folds × 50 epochs)


In [ ]:
# Run in foreground so that Kaggle's background commit ("Save & Run All") blocks and runs to completion.
# Expected runtime: ~1-1.5 hours.

!python3 -m training.train_dkt \
  --data data/training.csv --model dkt \
  --epochs 50 --folds 5 --out weights/dkt.pt \
  --batch 16 --device cuda --max-seq-len 2000

### CELL 6 — Graph-DKT full retrain (5 folds × 50 epochs)


In [ ]:
# Run in foreground.
# Expected runtime: ~3-4 hours.

!python3 -m training.train_dkt \
  --data data/training.csv --model graph_dkt \
  --epochs 50 --folds 5 --out weights/graph_dkt.pt \
  --batch 16 --device cuda --max-seq-len 2000

### CELL 7 — Compare results


In [ ]:
# Runs eval_folds.py to load fold0-4 weights for DKT vs Graph-DKT,
# evaluates them on their respective validation splits, and prints comparison table.
# CPU evaluation is used by default because the models are tiny (~146K parameters)
# and evaluating them on CPU is extremely fast (~2 minutes) and completely avoids CUDA VRAM OOM limits.
# We match the sequence truncation length to the training settings (--max-seq-len 2000).

!python3 -m training.eval_folds --segment-by-length --device cpu --max-seq-len 2000

### CELL 8 — Download results to your machine


In [ ]:
import zipfile, glob

with zipfile.ZipFile("/kaggle/working/results.zip", "w") as z:
    for f in glob.glob("weights/*.pt"):
        z.write(f)
    print("Packed weights:")
    for name in z.namelist():
        print(f"  {name}")

print("\nDone! Download /kaggle/working/results.zip from the Output panel of your Kaggle notebook.")